# ABSA Sentiment Model Pipeline

End-to-end pipeline covering aspect identification, sentiment model training, comparison, and evaluation.

Import dependencies.

In [1]:
import pandas as pd
import numpy as np
import joblib
import time
import tracemalloc
import os
import psutil
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from imblearn.over_sampling import SMOTE

## Hardware Utilities

Helper functions to track training time, inference time, and memory usage.

In [2]:
def get_process_memory_mb():
    """Get current process memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

def get_cpu_percent():
    """Get CPU usage percent."""
    return psutil.cpu_percent(interval=0.1)

def train_with_metrics(model, X_train, y_train, X_val, y_val, model_name):
    """Train a model and capture training time, inference time, and hardware usage."""
    mem_before = get_process_memory_mb()
    tracemalloc.start()
    
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train
    
    mem_after = get_process_memory_mb()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    start_infer = time.time()
    y_val_pred = model.predict(X_val)
    inference_time = time.time() - start_infer
    
    y_train_pred = model.predict(X_train)
    cpu_usage = get_cpu_percent()
    
    metrics = {
        'model_name': model_name,
        'model': model,
        'train_time': train_time,
        'inference_time': inference_time,
        'mem_before_mb': mem_before,
        'mem_after_mb': mem_after,
        'mem_used_mb': mem_after - mem_before,
        'peak_mem_mb': peak / (1024 * 1024),
        'cpu_percent': cpu_usage,
        'y_train_pred': y_train_pred,
        'y_val_pred': y_val_pred
    }
    
    return metrics

print(f"CPU cores: {psutil.cpu_count(logical=True)}")
print(f"Total RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"Available RAM: {psutil.virtual_memory().available / (1024**3):.1f} GB")

CPU cores: 8
Total RAM: 15.3 GB
Available RAM: 4.1 GB


## Part 1: Aspect Identification

Keyword-based aspect extraction using word-boundary regex matching.

In [3]:
import re

ASPECT_KEYWORDS = {
    'Network Coverage': [r'\bnetwork\b', r'\bcoverage\b', r'\bsignal\b', r'\bbars?\b', r'\bdead zone\b', r'\btower\b'],
    'Internet Speed': [r'\bspeed\b', r'\binternet\b', r'\bmbps\b', r'\bdownload\b', r'\bupload\b', r'\bbandwidth\b', r'\bbuffering\b'],
    'Call Quality': [r'\bcall quality\b', r'\bcall clarity\b', r'\bvoice\b', r'\bdropped call\b', r'\bcall drop\b', r'\becho\b', r'\bvolte\b', r'\bhd calling\b', r'\bcall\b'],
    'Customer Support': [r'\bcustomer support\b', r'\bcustomer care\b', r'\bcustomer service\b', r'\bhelpline\b', r'\bagent\b', r'\bcomplaint\b', r'\bticket\b', r'\bivr\b', r'\bsupport\b'],
    'Billing': [r'\bbilling\b', r'\bbill\b', r'\binvoice\b', r'\bcharge[sd]?\b', r'\bpayment\b', r'\bauto.?pay\b', r'\bdebit\b'],
    'Recharge Plans': [r'\brecharge\b', r'\bplan\b', r'\bpack\b', r'\bprepaid\b', r'\btopup\b'],
    'Data Balance': [r'\bdata balance\b', r'\bdata limit\b', r'\bdata usage\b', r'\bremaining data\b', r'\bdata cap\b', r'\b\d+\s*gb\b'],
    'Roaming': [r'\broaming\b', r'\btravel\b', r'\babroad\b'],
    'SIM Activation': [r'\bsim\b', r'\bactivation\b', r'\bekyc\b', r'\bnew sim\b'],
    'Mobile App Experience': [r'\bapp\b', r'\bmobile app\b', r'\bui\b', r'\binterface\b'],
    'OTT Bundle Services': [r'\bott\b', r'\bhotstar\b', r'\bnetflix\b', r'\bprime video\b', r'\bbundle\b'],
    'Pricing': [r'\bpric(e|ing)\b', r'\bcost\b', r'\bexpensive\b', r'\bcheap\b', r'\baffordable\b'],
    'Value for Money': [r'\bvalue for money\b', r'\bworth\b', r'\bvalue\b'],
    'Data Validity': [r'\bvalidity\b', r'\bexpir[ey]\b', r'\bvalid\b'],
    '5G Experience': [r'\b5g\b'],
    'Network Outage': [r'\boutage\b', r'\bmaintenance\b', r'\brestoration\b', r'\bdisruption\b', r'\bnetwork down\b'],
    'Number Portability': [r'\bportability\b', r'\bmnp\b', r'\bport(ed|ing)?\s+(my|the|a)?\s*number\b', r'\bnumber port\b'],
    'SMS Services': [r'\bsms\b', r'\botp\b', r'\btext message\b'],
    'Postpaid Plans': [r'\bpostpaid\b', r'\bmonthly plan\b'],
    'Network Congestion': [r'\bcongestion\b', r'\bpeak hour\b', r'\brush hour\b', r'\bthrottle\b'],
    'International Calling': [r'\binternational call\b', r'\bisd\b', r'\bstd\b', r'\babroad call\b'],
    'Device Compatibility': [r'\bdevice\b', r'\bcompatib\b', r'\bhandset\b', r'\bsamsung\b', r'\biphone\b']
}

def identify_aspects(text):
    """Identify aspects mentioned in feedback using word-boundary regex matching."""
    text_lower = text.lower()
    identified = []
    
    for aspect, patterns in ASPECT_KEYWORDS.items():
        for pattern in patterns:
            if re.search(pattern, text_lower):
                identified.append(aspect)
                break
    
    return identified if identified else ['General']

# Quick test
test_feedbacks = [
    "Network coverage is terrible in my area and call quality drops frequently.",
    "The 5G speed is amazing but the billing is confusing.",
    "Customer support took forever to respond about my roaming charges.",
    "Internet is okay for the price I pay."
]

for fb in test_feedbacks:
    print(f"{fb}\n  -> {identify_aspects(fb)}\n")

Network coverage is terrible in my area and call quality drops frequently.
  -> ['Network Coverage', 'Call Quality']

The 5G speed is amazing but the billing is confusing.
  -> ['Internet Speed', 'Billing', '5G Experience']

Customer support took forever to respond about my roaming charges.
  -> ['Customer Support', 'Billing', 'Roaming']

Internet is okay for the price I pay.
  -> ['Internet Speed', 'Pricing']



Evaluate aspect identification precision/recall on 500 labeled samples.

In [4]:
absa_raw = pd.read_csv("../data/raw/absa_dataset.csv")

feedback_aspects = absa_raw.groupby('feedback_id').agg({
    'feedback_text': 'first',
    'aspect': list
}).reset_index()

sample = feedback_aspects.sample(500, random_state=42)

correct = 0
total_true = 0
total_pred = 0

for _, row in sample.iterrows():
    true_aspects = set(row['aspect'])
    pred_aspects = set(identify_aspects(row['feedback_text']))
    
    correct += len(true_aspects & pred_aspects)
    total_true += len(true_aspects)
    total_pred += len(pred_aspects)

precision = correct / total_pred if total_pred > 0 else 0
recall = correct / total_true if total_true > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Aspect Identification (500 samples):")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")

Aspect Identification (500 samples):
  Precision: 0.7255
  Recall:    0.8422
  F1 Score:  0.7795


## Part 2: Sentiment Prediction

Load processed data, encode labels, split into train/val/test, and vectorize.

In [ ]:
absa_df = pd.read_csv("../data/processed/absa_processed.csv")

# Label encoding (previously in feature_eng notebook)
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
import hashlib

MODEL_VERSION = datetime.now().strftime('%Y%m%d_%H%M%S')
MODELS_DIR = os.path.join('..', 'models')
VERSIONED_DIR = os.path.join(MODELS_DIR, 'versions', MODEL_VERSION)
os.makedirs(VERSIONED_DIR, exist_ok=True)

def save_versioned_model(obj, filename, description=''):
    """Save model artifact with versioning metadata."""
    # Save to run path (latest)
    prod_path = os.path.join(MODELS_DIR, filename)
    joblib.dump(obj, prod_path)
    # Save versioned copy
    ver_path = os.path.join(VERSIONED_DIR, filename)
    joblib.dump(obj, ver_path)
    print(f"  Saved: {filename} -> {VERSIONED_DIR}/")
    return prod_path

label_encoder = LabelEncoder()
absa_df["sentiment_encoded"] = label_encoder.fit_transform(absa_df["sentiment"])
save_versioned_model(label_encoder, 'sentiment_label_encoder.pkl', 'LabelEncoder for sentiment classes')
print(f"\nModel version: {MODEL_VERSION}")
print(f"Label mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

print(f"\nDataset shape: {absa_df.shape}")
print("\nSentiment distribution:")
print(absa_df["sentiment"].value_counts())
print(f"\nAspect count: {absa_df['aspect'].nunique()}")

  Saved: sentiment_label_encoder.pkl -> ../models/versions/20260615_230059/

Model version: 20260615_230059
Label mapping: {'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}

Dataset shape: (13100, 8)

Sentiment distribution:
sentiment
Positive    4682
Neutral     4332
Negative    4086
Name: count, dtype: int64

Aspect count: 22


Filter to standard aspects and create combined ABSA input (aspect + text).

In [6]:
standard_aspects = [
    'Network Coverage', 'Call Quality', 'Internet Speed', 'Network Outage',
    'Billing', 'Customer Support', 'Roaming', 'Postpaid Plans',
    'SIM Activation', 'Network Congestion', 'International Calling',
    '5G Experience', 'OTT Bundle Services', 'Data Balance',
    'Mobile App Experience', 'Pricing', 'Device Compatibility',
    'Recharge Plans', 'SMS Services', 'Number Portability',
    'Data Validity', 'Value for Money'
]

clean_df = absa_df[absa_df["aspect"].isin(standard_aspects)].copy()
clean_df["absa_input"] = clean_df["aspect"].str.lower().str.replace(" ", "_") + " " + clean_df["processed_text"]

print(f"Rows with standard aspects: {len(clean_df)} / {len(absa_df)}")

Rows with standard aspects: 13100 / 13100


Group-aware train/val/test split (split by feedback_id to prevent data leakage).

In [7]:
# Split by feedback_id to prevent the same feedback text from appearing in both train and test.
# This avoids data leakage since one feedback_id can produce multiple aspect-level rows.
unique_ids = clean_df["feedback_id"].unique()

ids_train, ids_test = train_test_split(
    unique_ids, test_size=0.20, random_state=42
)
ids_train_split, ids_val = train_test_split(
    ids_train, test_size=0.2, random_state=42
)

train_mask = clean_df["feedback_id"].isin(ids_train_split)
val_mask = clean_df["feedback_id"].isin(ids_val)
test_mask = clean_df["feedback_id"].isin(ids_test)

X_train_split = clean_df.loc[train_mask, "absa_input"]
y_train_split = clean_df.loc[train_mask, "sentiment_encoded"]
X_val = clean_df.loc[val_mask, "absa_input"]
y_val = clean_df.loc[val_mask, "sentiment_encoded"]
X_test = clean_df.loc[test_mask, "absa_input"]
y_test = clean_df.loc[test_mask, "sentiment_encoded"]

print(f"Train: {len(X_train_split)} | Validation: {len(X_val)} | Test: {len(X_test)}")
print(f"Feedback IDs — Train: {len(ids_train_split)} | Val: {len(ids_val)} | Test: {len(ids_test)}")
print(f"\nNo feedback_id overlap: {len(set(ids_train_split) & set(ids_test)) == 0}")

Train: 8384 | Validation: 2096 | Test: 2620


Fit TF-IDF vectorizer and apply SMOTE oversampling.

In [8]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train_split)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_tfidf, y_train_split)

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"TF-IDF shapes — Train: {X_train_tfidf.shape}, Val: {X_val_tfidf.shape}, Test: {X_test_tfidf.shape}")
print(f"\nAfter SMOTE: {X_train_smote.shape}")
print(y_train_smote.value_counts().sort_index())

Vocabulary size: 11863
TF-IDF shapes — Train: (8384, 11863), Val: (2096, 11863), Test: (2620, 11863)

After SMOTE: (8991, 11863)
sentiment_encoded
0    2997
1    2997
2    2997
Name: count, dtype: int64


Define evaluation helper function.

In [9]:
label_names = ['Negative', 'Neutral', 'Positive']

def evaluate_model(model_name, y_true, y_pred, show_report=True):
    """Full evaluation with per-class metrics."""
    print(f"\n{'='*60}")
    print(f" {model_name}")
    print(f"{'='*60}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred, average='weighted'):.4f}")
    
    if show_report:
        print(f"\n{classification_report(y_true, y_pred, target_names=label_names, digits=4)}")
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    
    return f1_score(y_true, y_pred, average='weighted')

## Model Training & Comparison

Train Logistic Regression with balanced class weights.

In [10]:
lr_model = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    solver='lbfgs',
    max_iter=2000,
    random_state=42
)

lr_metrics = train_with_metrics(lr_model, X_train_tfidf, y_train_split, X_val_tfidf, y_val, "Logistic Regression")

print(f"Training time:  {lr_metrics['train_time']:.4f}s")
print(f"Inference time: {lr_metrics['inference_time']:.4f}s")
print(f"Peak memory:    {lr_metrics['peak_mem_mb']:.2f} MB")

lr_train_f1 = evaluate_model("LR (balanced) - TRAIN", y_train_split, lr_metrics['y_train_pred'])
lr_val_f1 = evaluate_model("LR (balanced) - VALIDATION", y_val, lr_metrics['y_val_pred'])

Training time:  7.8716s
Inference time: 0.0019s
Peak memory:    11.47 MB

 LR (balanced) - TRAIN
Accuracy:  0.8794
Precision: 0.8823
Recall:    0.8794
F1 Score:  0.8799

              precision    recall  f1-score   support

    Negative     0.8264    0.8994    0.8614      2615
     Neutral     0.9392    0.8694    0.9030      2772
    Positive     0.8785    0.8712    0.8749      2997

    accuracy                         0.8794      8384
   macro avg     0.8814    0.8800    0.8797      8384
weighted avg     0.8823    0.8794    0.8799      8384

Confusion Matrix:
[[2352   65  198]
 [ 199 2410  163]
 [ 295   91 2611]]

 LR (balanced) - VALIDATION
Accuracy:  0.8364
Precision: 0.8395
Recall:    0.8364
F1 Score:  0.8368

              precision    recall  f1-score   support

    Negative     0.7869    0.8471    0.8159       654
     Neutral     0.8979    0.8124    0.8530       693
    Positive     0.8314    0.8491    0.8402       749

    accuracy                         0.8364      2096
  

Train Naive Bayes on SMOTE-balanced data.

In [11]:
nb_model = MultinomialNB(alpha=0.5)

nb_metrics = train_with_metrics(nb_model, X_train_smote, y_train_smote, X_val_tfidf, y_val, "Naive Bayes")

print(f"Training time:  {nb_metrics['train_time']:.4f}s")
print(f"Inference time: {nb_metrics['inference_time']:.4f}s")
print(f"Peak memory:    {nb_metrics['peak_mem_mb']:.2f} MB")

nb_train_f1 = evaluate_model("NB (SMOTE) - TRAIN", y_train_smote, nb_metrics['y_train_pred'])
nb_val_f1 = evaluate_model("NB (SMOTE) - VALIDATION", y_val, nb_metrics['y_val_pred'])

Training time:  0.0120s
Inference time: 0.0013s
Peak memory:    1.37 MB

 NB (SMOTE) - TRAIN
Accuracy:  0.8650
Precision: 0.8747
Recall:    0.8650
F1 Score:  0.8658

              precision    recall  f1-score   support

    Negative     0.8107    0.9102    0.8576      2997
     Neutral     0.9813    0.8071    0.8858      2997
    Positive     0.8320    0.8775    0.8542      2997

    accuracy                         0.8650      8991
   macro avg     0.8747    0.8650    0.8658      8991
weighted avg     0.8747    0.8650    0.8658      8991

Confusion Matrix:
[[2728   20  249]
 [ 296 2419  282]
 [ 341   26 2630]]

 NB (SMOTE) - VALIDATION
Accuracy:  0.8397
Precision: 0.8495
Recall:    0.8397
F1 Score:  0.8406

              precision    recall  f1-score   support

    Negative     0.7746    0.8777    0.8229       654
     Neutral     0.9524    0.7792    0.8571       693
    Positive     0.8198    0.8625    0.8406       749

    accuracy                         0.8397      2096
   macro 

Train SGD-SVM with balanced class weights.

In [12]:
sgd_model = SGDClassifier(
    loss='modified_huber',
    class_weight='balanced',
    max_iter=5000,
    tol=1e-4,
    random_state=42
)

sgd_metrics = train_with_metrics(sgd_model, X_train_tfidf, y_train_split, X_val_tfidf, y_val, "SGD-SVM")

print(f"Training time:  {sgd_metrics['train_time']:.4f}s")
print(f"Inference time: {sgd_metrics['inference_time']:.4f}s")
print(f"Peak memory:    {sgd_metrics['peak_mem_mb']:.2f} MB")

sgd_train_f1 = evaluate_model("SGD-SVM (balanced) - TRAIN", y_train_split, sgd_metrics['y_train_pred'])
sgd_val_f1 = evaluate_model("SGD-SVM (balanced) - VALIDATION", y_val, sgd_metrics['y_val_pred'])

Training time:  0.6718s
Inference time: 0.0017s
Peak memory:    0.54 MB

 SGD-SVM (balanced) - TRAIN
Accuracy:  0.9099
Precision: 0.9122
Recall:    0.9099
F1 Score:  0.9104

              precision    recall  f1-score   support

    Negative     0.8601    0.9285    0.8930      2615
     Neutral     0.9613    0.9051    0.9324      2772
    Positive     0.9122    0.8982    0.9052      2997

    accuracy                         0.9099      8384
   macro avg     0.9112    0.9106    0.9102      8384
weighted avg     0.9122    0.9099    0.9104      8384

Confusion Matrix:
[[2428   40  147]
 [ 151 2509  112]
 [ 244   61 2692]]

 SGD-SVM (balanced) - VALIDATION
Accuracy:  0.8135
Precision: 0.8152
Recall:    0.8135
F1 Score:  0.8138

              precision    recall  f1-score   support

    Negative     0.7755    0.8028    0.7889       654
     Neutral     0.8651    0.8052    0.8341       693
    Positive     0.8036    0.8304    0.8168       749

    accuracy                         0.8135    

## Model Comparison Table

In [13]:
comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "SGD-SVM"],
    "Training Time (s)": [lr_metrics['train_time'], nb_metrics['train_time'], sgd_metrics['train_time']],
    "Inference Time (s)": [lr_metrics['inference_time'], nb_metrics['inference_time'], sgd_metrics['inference_time']],
    "Train F1": [lr_train_f1, nb_train_f1, sgd_train_f1],
    "Validation F1": [lr_val_f1, nb_val_f1, sgd_val_f1],
    "Train-Val Gap": [lr_train_f1 - lr_val_f1, nb_train_f1 - nb_val_f1, sgd_train_f1 - sgd_val_f1],
    "Peak Memory (MB)": [lr_metrics['peak_mem_mb'], nb_metrics['peak_mem_mb'], sgd_metrics['peak_mem_mb']],
    "CPU %": [lr_metrics['cpu_percent'], nb_metrics['cpu_percent'], sgd_metrics['cpu_percent']]
})

comparison_df.sort_values("Validation F1", ascending=False).reset_index(drop=True)

,Model,Training Time (s),Inference Time (s),Train F1,Validation F1,Train-Val Gap,Peak Memory (MB),CPU %
0,Naive Bayes,0.011963,0.001343,0.865841,0.840558,0.025282,1.366963,17.5
1,Logistic Regression,7.871555,0.001878,0.879944,0.836847,0.043097,11.472070,88.5
2,SGD-SVM,0.671791,0.001724,0.910362,0.813805,0.096556,0.535316,19.0


## Hyperparameter Tuning

Tune Logistic Regression using RandomizedSearchCV with 5-fold stratified CV.

In [14]:
param_distributions = {
    'C': [0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10, 50, 100],
    'solver': ['lbfgs', 'saga'],
    'penalty': ['l2'],
    'class_weight': ['balanced', None],
    'max_iter': [3000]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_search = RandomizedSearchCV(
    LogisticRegression(random_state=42),
    param_distributions,
    n_iter=20,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

lr_search.fit(X_train_smote, y_train_smote)

print(f"Best Parameters: {lr_search.best_params_}")
print(f"Best CV F1 Score: {lr_search.best_score_:.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


Best Parameters: {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 3000, 'class_weight': 'balanced', 'C': 0.5}
Best CV F1 Score: 0.8426


## Final Test Set Evaluation

Evaluate all models on the held-out test set.

In [15]:
best_lr = lr_search.best_estimator_

y_test_pred_lr = best_lr.predict(X_test_tfidf)
y_test_pred_nb = nb_metrics['model'].predict(X_test_tfidf)
y_test_pred_sgd = sgd_metrics['model'].predict(X_test_tfidf)

lr_test_f1 = evaluate_model("Tuned Logistic Regression - TEST", y_test, y_test_pred_lr)
nb_test_f1 = evaluate_model("Naive Bayes - TEST", y_test, y_test_pred_nb)
sgd_test_f1 = evaluate_model("SGD-SVM - TEST", y_test, y_test_pred_sgd)


 Tuned Logistic Regression - TEST
Accuracy:  0.8431
Precision: 0.8459
Recall:    0.8431
F1 Score:  0.8439

              precision    recall  f1-score   support

    Negative     0.7885    0.8397    0.8133       817
     Neutral     0.9186    0.8593    0.8880       867
    Positive     0.8285    0.8312    0.8299       936

    accuracy                         0.8431      2620
   macro avg     0.8452    0.8434    0.8437      2620
weighted avg     0.8459    0.8431    0.8439      2620

Confusion Matrix:
[[686  34  97]
 [ 58 745  64]
 [126  32 778]]

 Naive Bayes - TEST
Accuracy:  0.8397
Precision: 0.8488
Recall:    0.8397
F1 Score:  0.8414

              precision    recall  f1-score   support

    Negative     0.7756    0.8543    0.8130       817
     Neutral     0.9645    0.8155    0.8838       867
    Positive     0.8055    0.8494    0.8268       936

    accuracy                         0.8397      2620
   macro avg     0.8485    0.8397    0.8412      2620
weighted avg     0.8488    

Final summary comparison across all splits.

In [16]:
final_df = pd.DataFrame({
    "Model": ["Tuned LR", "Naive Bayes", "SGD-SVM"],
    "Train F1": [lr_train_f1, nb_train_f1, sgd_train_f1],
    "Val F1": [lr_val_f1, nb_val_f1, sgd_val_f1],
    "Test F1": [lr_test_f1, nb_test_f1, sgd_test_f1],
    "Train Time (s)": [lr_metrics['train_time'], nb_metrics['train_time'], sgd_metrics['train_time']],
    "Inference Time (s)": [lr_metrics['inference_time'], nb_metrics['inference_time'], sgd_metrics['inference_time']],
    "Memory (MB)": [lr_metrics['peak_mem_mb'], nb_metrics['peak_mem_mb'], sgd_metrics['peak_mem_mb']]
})

final_df.sort_values("Test F1", ascending=False).reset_index(drop=True)

,Model,Train F1,Val F1,Test F1,Train Time (s),Inference Time (s),Memory (MB)
0,Tuned LR,0.879944,0.836847,0.843918,7.871555,0.001878,11.472070
1,Naive Bayes,0.865841,0.840558,0.841369,0.011963,0.001343,1.366963
2,SGD-SVM,0.910362,0.813805,0.823293,0.671791,0.001724,0.535316


Neutral class performance comparison (critical for balanced ABSA).

In [17]:
for name, y_pred in [("Tuned LR", y_test_pred_lr), 
                      ("Naive Bayes", y_test_pred_nb),
                      ("SGD-SVM", y_test_pred_sgd)]:
    report = classification_report(y_test, y_pred, target_names=label_names, output_dict=True)
    neutral = report['Neutral']
    print(f"{name}: P={neutral['precision']:.4f}  R={neutral['recall']:.4f}  F1={neutral['f1-score']:.4f}")

Tuned LR: P=0.9186  R=0.8593  F1=0.8880
Naive Bayes: P=0.9645  R=0.8155  F1=0.8838
SGD-SVM: P=0.8790  R=0.8466  F1=0.8625


## Save Best Model & Vectorizer

In [18]:
print(f"\nModel version: {MODEL_VERSION}")
print(f"Versioned artifacts saved to: {VERSIONED_DIR}/")

save_versioned_model(best_lr, 'best_sentiment_model.pkl', 'Tuned Logistic Regression')
save_versioned_model(tfidf, 'tfidf_vectorizer.pkl', 'TF-IDF vectorizer (sentiment)')

# Save all models for Streamlit app model switching
save_versioned_model(nb_metrics['model'], 'naive_bayes_model.pkl', 'MultinomialNB (SMOTE)')
save_versioned_model(sgd_metrics['model'], 'sgd_svm_model.pkl', 'SGD-SVM (balanced)')

# Write version metadata
import json
metadata = {
    'version': MODEL_VERSION,
    'best_model': 'LogisticRegression',
    'best_model_params': best_lr.get_params(),
    'test_f1_lr': round(lr_test_f1, 4),
    'test_f1_nb': round(nb_test_f1, 4),
    'test_f1_sgd': round(sgd_test_f1, 4),
    'dataset_rows': len(clean_df),
    'train_size': len(X_train_split),
    'vocab_size': len(tfidf.vocabulary_),
    'imbalanced_learn_version': __import__('imblearn').__version__,
    'sklearn_version': __import__('sklearn').__version__
}
with open(os.path.join(VERSIONED_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"\nMetadata saved to: {VERSIONED_DIR}/metadata.json")
print(f"Best model params: {best_lr.get_params()}")


Model version: 20260615_230059
Versioned artifacts saved to: ../models/versions/20260615_230059/
  Saved: best_sentiment_model.pkl -> ../models/versions/20260615_230059/
  Saved: tfidf_vectorizer.pkl -> ../models/versions/20260615_230059/
  Saved: naive_bayes_model.pkl -> ../models/versions/20260615_230059/
  Saved: sgd_svm_model.pkl -> ../models/versions/20260615_230059/

Metadata saved to: ../models/versions/20260615_230059/metadata.json
Best model params: {'C': 0.5, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 3000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}


## End-to-End Demo

Full pipeline: raw text → aspect identification → sentence isolation → sentiment prediction.

In [19]:
import re
import nltk

# Ensure NLTK data is available
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(resource, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english')) - {'not', 'no', 'nor', 'never'}
noise_words = {'lol', 'bruh', 'fr', 'u', 'ur', 'pls', 'plz'}
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Apply full preprocessing pipeline to raw text."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and w not in noise_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return ' '.join(tokens)

def get_relevant_sentences(raw_text, aspect):
    """Extract only the sentences that mention the given aspect."""
    sentences = sent_tokenize(raw_text)
    patterns = ASPECT_KEYWORDS.get(aspect, [])
    
    relevant = []
    for sent in sentences:
        sent_lower = sent.lower()
        for pattern in patterns:
            if re.search(pattern, sent_lower):
                relevant.append(sent)
                break
    
    return ' '.join(relevant) if relevant else raw_text

def predict_absa(raw_text, model, vectorizer):
    """Full ABSA pipeline: identify aspects, isolate relevant sentences, predict sentiment."""
    aspects = identify_aspects(raw_text)
    sentiment_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    
    results = []
    for aspect in aspects:
        relevant_text = get_relevant_sentences(raw_text, aspect)
        processed = preprocess_text(relevant_text)
        absa_input = aspect.lower().replace(' ', '_') + ' ' + processed
        X = vectorizer.transform([absa_input])
        pred = model.predict(X)[0]
        results.append({'aspect': aspect, 'sentiment': sentiment_map[pred]})
    
    return results

# Demo
demo_feedbacks = [
    "Network coverage is pathetic in my area but call quality is decent when it works.",
    "The 5G speed is incredible! Best internet experience I've had. Though customer support could be better.",
    "Billing is okay nothing special. Standard charges as expected.",
    "Terrible roaming charges and the app crashes constantly. Worst experience ever."
]

for fb in demo_feedbacks:
    print(f"\nInput: \"{fb}\"")
    results = predict_absa(fb, best_lr, tfidf)
    for r in results:
        print(f"  \u2192 {r['aspect']}: {r['sentiment']}")


Input: "Network coverage is pathetic in my area but call quality is decent when it works."
  → Network Coverage: Neutral
  → Call Quality: Neutral

Input: "The 5G speed is incredible! Best internet experience I've had. Though customer support could be better."
  → Internet Speed: Positive
  → Customer Support: Negative
  → 5G Experience: Positive

Input: "Billing is okay nothing special. Standard charges as expected."
  → Billing: Neutral

Input: "Terrible roaming charges and the app crashes constantly. Worst experience ever."
  → Billing: Negative
  → Roaming: Negative
  → Mobile App Experience: Negative


## Part 3: Aspect Classifier (ML-based)

The keyword-based aspect detection misses implicit mentions like:
- "The mobile application takes too long to load" → keywords miss (no "app" keyword match)
- "Number transfer was smooth" → keywords miss (no "portability" keyword match)
- "Text messages arrive hours late" → keywords miss (no "sms" keyword match)

We train a multi-label classifier on the existing labeled data to detect aspects from context, not just keywords.

In [20]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

Group feedback by ID to create multi-label format (each feedback can have multiple aspects).

In [21]:
# Group by feedback to get multi-label format
feedback_aspects = absa_df.groupby('feedback_id').agg({
    'feedback_text': 'first',
    'aspect': list
}).reset_index()

print(f"Total unique feedback samples: {len(feedback_aspects)}")
print(f"Aspects per feedback (mean): {feedback_aspects['aspect'].apply(len).mean():.2f}")
print(f"Aspects per feedback (max): {feedback_aspects['aspect'].apply(len).max()}")

Total unique feedback samples: 6750
Aspects per feedback (mean): 1.94
Aspects per feedback (max): 4


Preprocess texts and encode aspect labels as binary matrix.

In [22]:
feedback_aspects['processed'] = feedback_aspects['feedback_text'].apply(preprocess_text)

# Multi-label binary encoding
mlb = MultiLabelBinarizer()
y_aspect = mlb.fit_transform(feedback_aspects['aspect'])
print(f"Aspect classes ({len(mlb.classes_)}): {list(mlb.classes_)}")

Aspect classes (22): ['5G Experience', 'Billing', 'Call Quality', 'Customer Support', 'Data Balance', 'Data Validity', 'Device Compatibility', 'International Calling', 'Internet Speed', 'Mobile App Experience', 'Network Congestion', 'Network Coverage', 'Network Outage', 'Number Portability', 'OTT Bundle Services', 'Postpaid Plans', 'Pricing', 'Recharge Plans', 'Roaming', 'SIM Activation', 'SMS Services', 'Value for Money']


Train/test split and TF-IDF vectorization with trigrams (trigrams help catch phrases like "number transfer", "text messages").

In [23]:
X_asp_train_text, X_asp_test_text, y_asp_train, y_asp_test = train_test_split(
    feedback_aspects['processed'], y_aspect,
    test_size=0.2, random_state=42
)

print(f"Train: {len(X_asp_train_text)} | Test: {len(X_asp_test_text)}")

# Separate TF-IDF for aspect detection (uses trigrams)
aspect_tfidf = TfidfVectorizer(
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=15000
)

X_asp_train_tfidf = aspect_tfidf.fit_transform(X_asp_train_text)
X_asp_test_tfidf = aspect_tfidf.transform(X_asp_test_text)
print(f"Vocabulary size: {len(aspect_tfidf.vocabulary_)}")

Train: 5400 | Test: 1350
Vocabulary size: 12340


Train OneVsRest Logistic Regression for multi-label aspect classification.

In [24]:
aspect_model = OneVsRestClassifier(
    LogisticRegression(
        C=2.0,
        solver='lbfgs',
        max_iter=3000,
        class_weight='balanced',
        random_state=42
    ),
    n_jobs=-1
)

aspect_model.fit(X_asp_train_tfidf, y_asp_train)
print("Aspect classifier trained.")

Aspect classifier trained.


Evaluate the aspect classifier.

In [25]:
from sklearn.metrics import f1_score as f1_metric

y_asp_pred = aspect_model.predict(X_asp_test_tfidf)

micro_f1 = f1_metric(y_asp_test, y_asp_pred, average='micro')
macro_f1 = f1_metric(y_asp_test, y_asp_pred, average='macro')
samples_f1 = f1_metric(y_asp_test, y_asp_pred, average='samples')

print(f"Micro F1: {micro_f1:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Samples F1: {samples_f1:.4f}")

print(f"\nPer-aspect performance:")
print(classification_report(y_asp_test, y_asp_pred, target_names=mlb.classes_, digits=3))

Micro F1: 0.9301
Macro F1: 0.9301
Samples F1: 0.9584

Per-aspect performance:
                       precision    recall  f1-score   support

        5G Experience      0.862     0.991     0.922       113
              Billing      0.938     0.938     0.938       129
         Call Quality      0.914     0.955     0.934       133
     Customer Support      0.876     0.968     0.920       124
         Data Balance      0.902     0.962     0.931       105
        Data Validity      0.891     0.968     0.928        93
 Device Compatibility      0.991     0.963     0.977       109
International Calling      0.891     0.975     0.931       118
       Internet Speed      0.849     0.975     0.908       121
Mobile App Experience      0.934     0.983     0.958       116
   Network Congestion      0.840     0.991     0.909       111
     Network Coverage      0.895     0.960     0.926       124
       Network Outage      0.892     0.976     0.932       127
   Number Portability      0.880     0.

Test on edge cases that keywords would miss.

In [26]:
edge_cases = [
    "The mobile application takes too long to load.",
    "Number transfer was smooth and hassle-free.",
    "Text messages arrive hours late.",
    "The porting process took forever.",
    "Streaming quality is terrible on this bundle.",
    "International travel connectivity was seamless.",
    "My phone is not supported by their network.",
    "The monthly subscription is overpriced.",
    "Data runs out too quickly.",
    "Call keeps dropping in my building.",
]

edge_processed = [preprocess_text(t) for t in edge_cases]
edge_tfidf = aspect_tfidf.transform(edge_processed)
edge_pred_proba = aspect_model.predict_proba(edge_tfidf)

for i, text in enumerate(edge_cases):
    probs = edge_pred_proba[i]
    top_indices = np.argsort(probs)[::-1][:3]
    top_aspects = [(mlb.classes_[idx], f"{probs[idx]:.2f}") for idx in top_indices if probs[idx] > 0.3]
    keyword_aspects = identify_aspects(text)
    
    print(f'\n"{text}"')
    print(f"  Keywords:   {keyword_aspects}")
    print(f"  Classifier: {top_aspects}")


"The mobile application takes too long to load."
  Keywords:   ['General']
  Classifier: [('Mobile App Experience', '0.89')]

"Number transfer was smooth and hassle-free."
  Keywords:   ['General']
  Classifier: [('Number Portability', '0.79'), ('SIM Activation', '0.36')]

"Text messages arrive hours late."
  Keywords:   ['General']
  Classifier: [('SMS Services', '0.88'), ('Network Congestion', '0.32')]

"The porting process took forever."
  Keywords:   ['General']
  Classifier: [('Number Portability', '0.95')]

"Streaming quality is terrible on this bundle."
  Keywords:   ['OTT Bundle Services']
  Classifier: [('OTT Bundle Services', '0.98'), ('Call Quality', '0.50')]

"International travel connectivity was seamless."
  Keywords:   ['Roaming']
  Classifier: [('International Calling', '0.68')]

"My phone is not supported by their network."
  Keywords:   ['Network Coverage']
  Classifier: [('Device Compatibility', '0.80')]

"The monthly subscription is overpriced."
  Keywords:   ['Gen

## Save Aspect Classifier

In [27]:
save_versioned_model(aspect_model, 'aspect_classifier.pkl', 'OneVsRest LR aspect classifier')
save_versioned_model(mlb, 'aspect_mlb.pkl', 'MultiLabelBinarizer for aspects')
save_versioned_model(aspect_tfidf, 'aspect_tfidf.pkl', 'TF-IDF vectorizer (aspect)')

print(f"\nAspect classifier version: {MODEL_VERSION}")

  Saved: aspect_classifier.pkl -> ../models/versions/20260615_230059/
  Saved: aspect_mlb.pkl -> ../models/versions/20260615_230059/
  Saved: aspect_tfidf.pkl -> ../models/versions/20260615_230059/

Aspect classifier version: 20260615_230059


## Part 4: DistilBERT Fine-Tuning

Fine-tune DistilBERT on the ABSA dataset for aspect-level sentiment classification.

**Key differences from TF-IDF models:**
- Uses raw text (no preprocessing needed — BERT handles context, word order, negation natively)
- Input format: `[Aspect Name] raw feedback text`
- Captures semantic meaning rather than just word presence


In [28]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


Prepare data with BERT input format (raw text, no TF-IDF preprocessing).

In [29]:
# Reload data for BERT (uses raw feedback_text, not processed_text)
bert_df = pd.read_csv("../data/processed/absa_processed.csv")
bert_df["sentiment_encoded"] = label_encoder.transform(bert_df["sentiment"])
bert_df["bert_input"] = "[" + bert_df["aspect"] + "] " + bert_df["feedback_text"]

# Use the same feedback_id-based split to prevent data leakage
X_b_train_split = bert_df.loc[bert_df["feedback_id"].isin(ids_train_split), "bert_input"].values
y_b_train_split = bert_df.loc[bert_df["feedback_id"].isin(ids_train_split), "sentiment_encoded"].values
X_b_val = bert_df.loc[bert_df["feedback_id"].isin(ids_val), "bert_input"].values
y_b_val = bert_df.loc[bert_df["feedback_id"].isin(ids_val), "sentiment_encoded"].values
X_b_test = bert_df.loc[bert_df["feedback_id"].isin(ids_test), "bert_input"].values
y_b_test = bert_df.loc[bert_df["feedback_id"].isin(ids_test), "sentiment_encoded"].values

print(f"Train: {len(X_b_train_split)} | Val: {len(X_b_val)} | Test: {len(X_b_test)}")
print(f"\nSample input: {X_b_train_split[0][:100]}...")

Train: 8384 | Val: 2096 | Test: 2620

Sample input: [International Calling] International calling is the weakest point. Number portability to a provider...


In [30]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class ABSADataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True,
            return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

BATCH_SIZE = 32
train_dataset = ABSADataset(X_b_train_split, y_b_train_split, tokenizer)
val_dataset = ABSADataset(X_b_val, y_b_val, tokenizer)
test_dataset = ABSADataset(X_b_test, y_b_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

Batches — Train: 262 | Val: 66 | Test: 82


Load pre-trained DistilBERT and fine-tune for 4 epochs.

In [31]:
bert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3
)
bert_model = bert_model.to(device)

EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(bert_model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

print(f"Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")
print(f"Total steps: {total_steps}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1430.84it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 66,955,779
Total steps: 786


In [32]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    preds_all, labels_all = [], []
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        preds_all.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(labels_all, preds_all, average='weighted')

def evaluate_bert(model, loader, device):
    model.eval()
    total_loss = 0
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds_all.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(labels_all, preds_all, average='weighted'), preds_all, labels_all

In [33]:
best_val_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    train_loss, train_f1 = train_epoch(bert_model, train_loader, optimizer, scheduler, device)
    val_loss, val_f1, _, _ = evaluate_bert(bert_model, val_loader, device)
    
    print(f"Epoch {epoch+1}/{EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val F1:   {val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in bert_model.state_dict().items()}
        print(f"  -> New best (Val F1: {val_f1:.4f})")
    print()

bert_model.load_state_dict(best_state)
print(f"Best validation F1: {best_val_f1:.4f}")

Epoch 1/3:
  Train Loss: 0.5666 | Train F1: 0.7663
  Val Loss:   0.1985 | Val F1:   0.9338
  -> New best (Val F1: 0.9338)

Epoch 2/3:
  Train Loss: 0.1565 | Train F1: 0.9514
  Val Loss:   0.1381 | Val F1:   0.9595
  -> New best (Val F1: 0.9595)

Epoch 3/3:
  Train Loss: 0.0969 | Train F1: 0.9700
  Val Loss:   0.1336 | Val F1:   0.9594

Best validation F1: 0.9595


Evaluate DistilBERT on the test set.

In [34]:
test_loss, test_f1, test_preds, test_true = evaluate_bert(bert_model, test_loader, device)

print(f"DistilBERT Test F1: {test_f1:.4f}")
print(f"\n{classification_report(test_true, test_preds, target_names=label_names, digits=4)}")

DistilBERT Test F1: 0.9565

              precision    recall  f1-score   support

    Negative     0.9307    0.9865    0.9578       817
     Neutral     0.9715    0.9435    0.9573       867
    Positive     0.9671    0.9423    0.9545       936

    accuracy                         0.9565      2620
   macro avg     0.9564    0.9574    0.9565      2620
weighted avg     0.9572    0.9565    0.9565      2620



Save DistilBERT model and tokenizer for the Streamlit app.

In [35]:
save_path = os.path.join(MODELS_DIR, 'distilbert_sentiment')
os.makedirs(save_path, exist_ok=True)

bert_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Also save versioned copy of DistilBERT
ver_bert_path = os.path.join(VERSIONED_DIR, 'distilbert_sentiment')
os.makedirs(ver_bert_path, exist_ok=True)
bert_model.save_pretrained(ver_bert_path)
tokenizer.save_pretrained(ver_bert_path)

total_size = sum(os.path.getsize(os.path.join(save_path, f)) for f in os.listdir(save_path))
print(f"Saved DistilBERT to: {save_path}/")
print(f"Versioned copy: {ver_bert_path}/")
print(f"Model size: {total_size / (1024*1024):.1f} MB")
print(f"Test F1: {test_f1:.4f}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.85it/s]

Saved DistilBERT to: ../models/distilbert_sentiment/
Versioned copy: ../models/versions/20260615_230059/distilbert_sentiment/
Model size: 256.1 MB
Test F1: 0.9565


Quick inference demo with DistilBERT.

In [36]:
def predict_bert_single(text, aspect, model, tokenizer, device):
    model.eval()
    input_text = f"[{aspect}] {text}"
    encoding = tokenizer(
        input_text, add_special_tokens=True, max_length=128,
        padding='max_length', truncation=True, return_tensors='pt'
    )
    with torch.no_grad():
        outputs = model(
            input_ids=encoding['input_ids'].to(device),
            attention_mask=encoding['attention_mask'].to(device)
        )
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    sentiment_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    pred = int(np.argmax(probs))
    return sentiment_map[pred], float(probs[pred])

# Compare DistilBERT vs Logistic Regression
test_cases = [
    ("Network is very bad but pricing is good", "Network Coverage"),
    ("Network is very bad but pricing is good", "Pricing"),
    ("Internet speed is amazing, best service ever", "Internet Speed"),
    ("Call quality is trash, sounds like talking underwater", "Call Quality"),
    ("Billing is okay nothing special", "Billing"),
]

print(f"{'Input':<55} {'Aspect':<20} {'BERT':<20} {'LR (TF-IDF)'}")
print("=" * 110)
for text, aspect in test_cases:
    bert_sent, bert_conf = predict_bert_single(text, aspect, bert_model, tokenizer, device)
    # LR prediction for comparison
    processed = preprocess_text(text)
    absa_input = aspect.lower().replace(' ', '_') + ' ' + processed
    lr_pred = best_lr.predict(tfidf.transform([absa_input]))[0]
    lr_sent = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}[lr_pred]
    print(f"{text:<55} {aspect:<20} {bert_sent} ({bert_conf:.0%}){'':5} {lr_sent}")

Input                                                   Aspect               BERT                 LR (TF-IDF)
Network is very bad but pricing is good                 Network Coverage     Negative (49%)      Positive
Network is very bad but pricing is good                 Pricing              Positive (83%)      Positive
Internet speed is amazing, best service ever            Internet Speed       Positive (99%)      Positive
Call quality is trash, sounds like talking underwater   Call Quality         Negative (98%)      Negative
Billing is okay nothing special                         Billing              Neutral (99%)      Neutral


## Final Model Comparison (All 4 Models)

Compare DistilBERT against all sklearn models on the same test set.

In [37]:
import time

# Measure DistilBERT inference time
start_infer = time.time()
_, _, bert_test_preds, bert_test_true = evaluate_bert(bert_model, test_loader, device)
bert_inference_time = time.time() - start_infer

# Get model size
bert_size_mb = sum(os.path.getsize(os.path.join(save_path, f)) for f in os.listdir(save_path)) / (1024*1024)

# Final comparison table including DistilBERT
all_models_df = pd.DataFrame({
    "Model": ["Tuned LR", "Naive Bayes", "SGD-SVM", "DistilBERT"],
    "Train F1": [lr_train_f1, nb_train_f1, sgd_train_f1, "—"],
    "Val F1": [lr_val_f1, nb_val_f1, sgd_val_f1, f"{best_val_f1:.4f}"],
    "Test F1": [lr_test_f1, nb_test_f1, sgd_test_f1, test_f1],
    "Inference Time (s)": [
        lr_metrics['inference_time'], nb_metrics['inference_time'],
        sgd_metrics['inference_time'], bert_inference_time
    ],
    "Model Size (MB)": [
        lr_metrics['peak_mem_mb'], nb_metrics['peak_mem_mb'],
        sgd_metrics['peak_mem_mb'], bert_size_mb
    ]
})

print("="*80)
print(" FINAL MODEL COMPARISON — All 4 Models")
print("="*80)
all_models_df.sort_values("Test F1", ascending=False).reset_index(drop=True)

 FINAL MODEL COMPARISON — All 4 Models


,Model,Train F1,Val F1,Test F1,Inference Time (s),Model Size (MB)
0,DistilBERT,—,0.9595,0.956471,233.623559,256.107767
1,Tuned LR,0.879944,0.836847,0.843918,0.001878,11.472070
2,Naive Bayes,0.865841,0.840558,0.841369,0.001343,1.366963
3,SGD-SVM,0.910362,0.813805,0.823293,0.001724,0.535316


In [38]:
# Per-class comparison: DistilBERT vs Best sklearn (Tuned LR)
print("\n" + "="*80)
print(" PER-CLASS COMPARISON: DistilBERT vs Tuned Logistic Regression")
print("="*80)

print("\n--- DistilBERT ---")
print(classification_report(bert_test_true, bert_test_preds, target_names=label_names, digits=4))

print("--- Tuned Logistic Regression ---")
print(classification_report(y_test, y_test_pred_lr, target_names=label_names, digits=4))

# Highlight improvements
bert_report = classification_report(bert_test_true, bert_test_preds, target_names=label_names, output_dict=True)
lr_report = classification_report(y_test, y_test_pred_lr, target_names=label_names, output_dict=True)

print("\n--- Improvement (BERT - LR) ---")
for cls in label_names:
    diff = bert_report[cls]['f1-score'] - lr_report[cls]['f1-score']
    arrow = '↑' if diff > 0 else '↓' if diff < 0 else '→'
    print(f"  {cls}: {arrow} {diff:+.4f} F1")


 PER-CLASS COMPARISON: DistilBERT vs Tuned Logistic Regression

--- DistilBERT ---
              precision    recall  f1-score   support

    Negative     0.9307    0.9865    0.9578       817
     Neutral     0.9715    0.9435    0.9573       867
    Positive     0.9671    0.9423    0.9545       936

    accuracy                         0.9565      2620
   macro avg     0.9564    0.9574    0.9565      2620
weighted avg     0.9572    0.9565    0.9565      2620

--- Tuned Logistic Regression ---
              precision    recall  f1-score   support

    Negative     0.7885    0.8397    0.8133       817
     Neutral     0.9186    0.8593    0.8880       867
    Positive     0.8285    0.8312    0.8299       936

    accuracy                         0.8431      2620
   macro avg     0.8452    0.8434    0.8437      2620
weighted avg     0.8459    0.8431    0.8439      2620


--- Improvement (BERT - LR) ---
  Negative: ↑ +0.1445 F1
  Neutral: ↑ +0.0693 F1
  Positive: ↑ +0.1247 F1
